In [9]:
import csv
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

with open("wine_dataset.csv", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    features = [name for name in reader.fieldnames if name != "target"]

X = torch.tensor([[float(row[c]) for c in features] for row in rows], dtype=torch.float32)
y = torch.tensor([int(row["target"]) for row in rows], dtype=torch.long)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("classes:", torch.unique(y, return_counts=True))
assert X.shape == (178, 13) and y.shape == (178,)

X: torch.Size([178, 13]) torch.float32
y: torch.Size([178]) torch.int64
classes: (tensor([0, 1, 2]), tensor([59, 71, 48]))


In [10]:
def stratified_split(y, test_fraction=0.2, seed=42):
    g = torch.Generator().manual_seed(seed)
    train_parts, test_parts = [], []
    for cls in torch.unique(y):
        indices = torch.where(y == cls)[0]
        indices = indices[torch.randperm(indices.numel(), generator=g)]
        n_test = round(indices.numel() * test_fraction)
        test_parts.append(indices[:n_test])
        train_parts.append(indices[n_test:])
    return torch.cat(train_parts), torch.cat(test_parts)

train_idx, test_idx = stratified_split(y)
assert len(train_idx) + len(test_idx) == len(y)
assert not set(train_idx.tolist()) & set(test_idx.tolist())
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("train/test:", len(train_idx), len(test_idx))
print("train class distribution:", torch.bincount(y_train))
print("test class distribution:", torch.bincount(y_test))

train/test: 142 36
train class distribution: tensor([47, 57, 38])
test class distribution: tensor([12, 14, 10])


In [11]:
def standardize(train_x, val_x):
    mean = train_x.mean(dim=0)
    std = train_x.std(dim=0, unbiased=False).clamp_min(1e-8)
    return (train_x - mean) / std, (val_x - mean) / std

xtr, xte = standardize(X_train, X_test)
assert xtr.shape == X_train.shape and xte.shape == X_test.shape
assert torch.all(torch.isfinite(xtr)) and torch.all(torch.isfinite(xte))
assert torch.allclose(xtr.mean(0), torch.zeros(13), atol=1e-5)
assert torch.allclose(xtr.std(0, unbiased=False), torch.ones(13), atol=1e-5)

In [12]:
def dummy_accuracy(y_train, y_other):
    majority_class = torch.bincount(y_train).argmax()
    predictions = torch.full_like(y_other, majority_class.item())
    return (predictions == y_other).float().mean().item()

print("Dummy train accuracy:",
      dummy_accuracy(y_train, y_train))

Dummy train accuracy (только самопроверка): 0.40140846371650696


In [13]:
torch.manual_seed(42)
model = nn.Linear(13, 3)
assert model(torch.randn(5, 13)).shape == (5, 3)

In [14]:
def fit(train_x, train_y, epochs=250, seed=42):
    torch.manual_seed(seed)

    model = nn.Linear(13, 3)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

    dataset = TensorDataset(train_x, train_y)
    loader = DataLoader(dataset, batch_size=16, shuffle=True)

    for epoch in range(epochs):
        model.train()

        for xb, yb in loader:
            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

    return model

In [15]:
@torch.no_grad()
def accuracy(model, x, y):
    model.eval()

    logits = model(x)
    predictions = logits.argmax(dim=1)

    return (predictions == y).float().mean().item()

In [16]:
def make_stratified_folds(y, k=5, seed=42):
    g = torch.Generator().manual_seed(seed)
    val_parts = [[] for _ in range(k)]

    for cls in torch.unique(y):
        indices = torch.where(y == cls)[0]
        indices = indices[torch.randperm(indices.numel(), generator=g)]
        for j, chunk in enumerate(torch.tensor_split(indices, k)):
            val_parts[j].append(chunk)

    val_folds = [torch.cat(parts) for parts in val_parts]
    return [(torch.cat([v for j, v in enumerate(val_folds) if j != i]), val_folds[i]) for i in range(k)]

folds = make_stratified_folds(y_train)
all_valid = torch.cat([val for _, val in folds])
assert len(folds) == 5
assert sorted(all_valid.tolist()) == list(range(len(y_train)))
for tr, val in folds:
    assert not set(tr.tolist()) & set(val.tolist())

In [17]:
cv_scores = []
dummy_scores = []
for fold_number, (tr_idx, val_idx) in enumerate(folds, start=1):
    xtr, xval = standardize(X_train[tr_idx], X_train[val_idx])
    model = fit(xtr, y_train[tr_idx])
    score = accuracy(model, xval, y_train[val_idx])
    baseline = dummy_accuracy(y_train[tr_idx], y_train[val_idx])
    cv_scores.append(score)
    dummy_scores.append(baseline)
    print(f"fold {fold_number}: torch={score:.3f}, dummy={baseline:.3f}")

scores = torch.tensor(cv_scores)
print("CV mean:", scores.mean().item())
print("CV std:", scores.std(unbiased=False).item())
print("dummy mean:", torch.tensor(dummy_scores).mean().item())

fold 1: torch=1.000, dummy=0.400
fold 2: torch=1.000, dummy=0.400
fold 3: torch=0.964, dummy=0.393
fold 4: torch=0.963, dummy=0.407
fold 5: torch=1.000, dummy=0.407
CV mean: 0.9854497909545898
CV std: 0.017825260758399963
dummy mean: 0.40153437852859497


In [18]:
xtr, xte = standardize(X_train, X_test)
final_model = fit(xtr, y_train)
print("Final test accuracy:", accuracy(final_model, xte, y_test))

Final test accuracy: 0.9722222089767456
